In [1]:
%load_ext autoreload
%autoreload 2

from seek_code import SEEK
from backbone import Backbone
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN

from pathlib import Path
from torch.utils.data import DataLoader, Subset

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = EleHandler(subset='IDI_6')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

code = 'debug_rentree'


### Baseline 1 - MegaDescriptor out of the box, with savannah elephants pretraining

In [3]:
code = 'refresher(10epochs)-baseline-1'

MD_savannah = Backbone(model_name="MegaDescriptor", pretraining="savannah_elephants", experiment_code=code)
r = Retrieval(experiment_code = code)
r.evaluate_model(model = MD_savannah, train_loader=train_loader, test_loader=test_loader)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

evaluating backbone


Train Recalls:
Recall@1: 6.71%
Recall@5: 12.52%
Recall@10: 15.90%
Recall@20: 21.20%
Recall@100: 45.18%
Test Recalls:
Recall@1: 23.15%
Recall@5: 32.98%
Recall@10: 38.08%
Recall@20: 45.16%
Recall@100: 67.71%


{1: 0.23153084539223154,
 5: 0.32977913175932977,
 10: 0.38080731150038083,
 20: 0.45163747143945165,
 100: 0.6770753998476771}

23.15% with IDI_6 and savannah_elephants weights

## Baseline 2 - MD finetuned

In [4]:
code = 'refresher(10epochs)-baseline-2'

MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
r = Retrieval(experiment_code = code)
r.evaluate_model(model = MD_finetuned, train_loader=train_loader, test_loader=test_loader)

evaluating backbone
Train Recalls:
Recall@1: 53.15%
Recall@5: 75.01%
Recall@10: 82.89%
Recall@20: 89.10%
Recall@100: 97.83%
Test Recalls:
Recall@1: 38.39%
Recall@5: 54.23%
Recall@10: 60.40%
Recall@20: 66.57%
Recall@100: 83.70%


{1: 0.38385376999238385,
 5: 0.5422696115765423,
 10: 0.6039603960396039,
 20: 0.6656511805026657,
 100: 0.837014470677837}

38.38% with IDI_6 and MD_finetuned weights

## Exp 1 : Projector replicates without any correction

For 100 epochs, I train without any correction at trainning time and I test it under several correction policies.

In [5]:
code = 'refresher(10epochs)-exp1'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="savannah_elephants", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=5e-6, scale=64, margin=0.5, experiment_code=code, intervention_fn=None, alpha = 0)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.unfreeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction": None,
    "0% Correction Hard": SEEK.hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

wandb: Currently logged in as: antoinesalaun (antoinesalaun-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:388: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: True, Backbone_for_concepts: False, Backbone: True, Concept_head: False
Epoch 1/100 | Loss: 7.1446 | Batch R@1: 0.91% | Train: R@1=6.2% R@5=12.0% R@20=20.8% R@100=43.8% | Val: R@1=22.2% R@5=32.1% R@10=36.6% R@20=43.6% R@100=64.9%
Epoch 6/100 | Loss: 6.4220 | Batch R@1: 0.66% | Train: R@1=6.1% R@5=12.2% R@20=20.9% R@100=44.8% | Val: R@1=22.6% R@5=32.7% R@10=37.2% R@20=43.9% R@100=64.7%
Epoch 11/100 | Loss: 6.0544 | Batch R@1: 0.25% | Train: R@1=6.8% R@5=12.2% R@20=21.7% R@100=46.8% | Val: R@1=23.6% R@5=34.0% R@10=39.4% R@20=46.3% R@100=67.0%
Epoch 16/100 | Loss: 5.6491 | Batch R@1: 0.45% | Train: R@1=7.3% R@5=12.9% R@20=23.2% R@100=48.7% | Val: R@1=25.6% R@5=36.4% R@10=42.1% R@20=48.8% R@100=71.6%
Epoch 21/100 | Loss: 5.1909 | Batch R@1: 1.06% | Train: R@1=8.2% R@5=15.0% R@20=25.5% R@100=51.7% | Val: R@1=28.0% R@5=38.9% R@10=

BATCH-Recall@1,▁▂▂▁▂▂▁▂▃▃▃▄▃▃▄▃▄▃▄▄▆▄▃▅▄▄▅▄▆▄▇▆▅█▆▇█▆██
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train-Recall@1,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇██
train-Recall@100,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇█████
train-Recall@20,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train-Recall@5,▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇█
train_loss,███▇▇▇▇▆▆▆▆▆▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁
val-Recall@1,▁▁▂▃▅▆▆▆▇▇▆▇▇▇▇▇▇▇██
val-Recall@10,▁▁▂▄▅▇▇█████████████
val-Recall@100,▁▁▂▄▅▆▇███████████▇█
val-Recall@20,▁▁▂▃▅▆▇▇███▇████████


{'100% Correction': {1: 0.32444782939832445,
  5: 0.44554455445544555,
  10: 0.507996953541508,
  20: 0.5887281035795887,
  100: 0.8057882711348058},
 '50% Correction + soft': {1: 0.32444782939832445,
  5: 0.44554455445544555,
  10: 0.507996953541508,
  20: 0.5887281035795887,
  100: 0.8057882711348058},
 '0% Correction': {1: 0.32444782939832445,
  5: 0.44554455445544555,
  10: 0.507996953541508,
  20: 0.5887281035795887,
  100: 0.8057882711348058},
 '0% Correction Hard': {1: 0.32444782939832445,
  5: 0.44554455445544555,
  10: 0.507996953541508,
  20: 0.5887281035795887,
  100: 0.8057882711348058}}

je dois comprendre pourquoi la correction n'affecte pas le test ?

##  Exp-Proof of concept : training and testing with perfect concepts

In [6]:
code = 'refresher(10epochs)-PoC'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=5e-6, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.oracle_correction, alpha = 0)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.unfreeze()
concept_head.freeze()
pr.unfreeze()

# Train the model with perfect oracle corrections
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100)

pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:388: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: True, Backbone_for_concepts: False, Backbone: True, Concept_head: False
Epoch 1/100 | Loss: 7.0947 | Batch R@1: 6.77% | Train: R@1=52.1% R@5=74.4% R@20=88.7% R@100=97.6% | Val: R@1=38.2% R@5=53.8% R@10=60.0% R@20=66.6% R@100=83.6%
Epoch 6/100 | Loss: 6.1595 | Batch R@1: 5.95% | Train: R@1=48.1% R@5=71.1% R@20=86.9% R@100=96.8% | Val: R@1=37.9% R@5=52.2% R@10=59.3% R@20=64.9% R@100=82.9%
Epoch 11/100 | Loss: 5.3610 | Batch R@1: 6.97% | Train: R@1=48.8% R@5=71.5% R@20=87.4% R@100=97.0% | Val: R@1=37.1% R@5=51.7% R@10=57.7% R@20=64.9% R@100=82.3%
Epoch 16/100 | Loss: 4.6237 | Batch R@1: 5.50% | Train: R@1=52.1% R@5=73.9% R@20=88.3% R@100=97.1% | Val: R@1=36.3% R@5=51.3% R@10=57.6% R@20=64.3% R@100=82.6%
Epoch 21/100 | Loss: 3.9388 | Batch R@1: 6.56% | Train: R@1=54.8% R@5=76.2% R@20=89.6% R@100=97.5% | Val: R@1=36.5% R@5=50.5% 

BATCH-Recall@1,▂▁▁▂▂▁▃▂▂▄▅▄▃▅▅▅▄▄▅▅▄▆▅▆▃▅▅▇▆▇▆▆▆█▆▇█▆▇▆
epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train-Recall@1,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▇▇▇▇▇▇▇▇▇█████
train-Recall@100,▃▂▁▁▂▃▂▂▃▄▅▅▅▅▅▆▆▇▆▇▇▇▇▇▇███████████████
train-Recall@20,▂▁▁▁▁▁▁▂▂▂▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
train-Recall@5,▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
train_loss,██▇▇▇▆▅▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val-Recall@1,█▇▆▅▆▅▄▄▃▃▂▂▂▂▂▂▂▁▁▁
val-Recall@10,█▇▅▅▅▃▃▃▂▃▃▂▂▂▂▁▂▁▁▁
val-Recall@100,█▇▆▆▆▇▆▆▅▅▄▄▃▃▃▂▁▁▁▁
val-Recall@20,█▆▆▅▅▄▄▃▂▂▂▂▂▁▂▁▁▁▁▁


{'100% Correction': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '50% Correction + soft': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '0% Correction': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '0% Correction Hard': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905}}

In [9]:

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction": None,
    "0% Correction Hard": SEEK.hard
}
# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

Retrieval with 100% Correction correction-------------------------------------
evaluating projector  with intervention
concept logits :  tensor([ 5.0565, -4.2426, -3.0529,  3.2190, -3.4201, -1.6105, -0.4018,  4.4276,
        -2.3889, -1.0846,  1.5328, -0.2859, -3.5951, -0.7900,  0.1660,  3.2378,
         4.5412, -3.0964, -0.9469, -0.3812,  0.1427,  1.3959, -4.5062, -1.4042,
         1.9205, -1.3328,  5.1123, -1.9288, -1.7471, -1.0824, -1.3872,  1.0971,
        -3.4391,  1.4913,  0.3366, -0.9766,  1.8197, -2.4426,  0.5811, -0.1611,
        -0.7207,  1.2839, -3.3301, -0.0475, -0.4115, -0.3400,  3.0330, -2.8994,
        -1.5505, -1.9844, -1.5962,  2.1462,  2.2787, -4.5868,  1.8870,  1.5517,
        -5.3536,  4.0120, -2.6510, -6.8237,  3.9202, -1.1499, -6.0829],
       device='cuda:0')  edited_concepts :  tensor([1., 0., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 1., 0., 0.,

{'100% Correction': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '50% Correction + soft': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '0% Correction': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905},
 '0% Correction Hard': {1: 0.3351104341203351,
  5: 0.4645849200304646,
  10: 0.5331302361005331,
  20: 0.6016755521706016,
  100: 0.7905559786747905}}

## Checking that it works

Previous results show no influence of the correciton function so I am a bit worried that there is a bug somewhere !

I will start by printing the size of embeedings first.

In [3]:
code = 'refresher(10epochs)-exp1'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="savannah_elephants", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=5e-6, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.oracle_correction, alpha = 0.5, print_every = 1)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.unfreeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=5)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction": None,
    "0% Correction Hard": SEEK.hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts,
    backbone=MD_finedtuned,
    concept_head=concept_head,
    train_loader=train_loader,
    test_loader=test_loader,
    intervention_fns=test_intervention_fns
)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:388: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: True, Backbone_for_concepts: False, Backbone: True, Concept_head: False
Embeddings norm: 46.2105712890625, Projected concepts norm: 55.070743560791016
Epoch 1/5 | Loss: 7.0001 | Batch R@1: 1.31% | Train: R@1=16.8% R@5=32.8% R@20=49.9% R@100=71.9% | Val: R@1=31.2% R@5=42.1% R@10=48.2% R@20=54.8% R@100=74.1%
Embeddings norm: 43.136199951171875, Projected concepts norm: 56.73817443847656
Epoch 2/5 | Loss: 6.7626 | Batch R@1: 1.01% | Train: R@1=17.6% R@5=33.6% R@20=51.0% R@100=72.9% | Val: R@1=31.8% R@5=42.7% R@10=48.5% R@20=55.3% R@100=74.5%
Embeddings norm: 44.288307189941406, Projected concepts norm: 56.65456771850586
Epoch 3/5 | Loss: 6.6107 | Batch R@1: 2.07% | Train: R@1=17.9% R@5=34.3% R@20=50.9% R@100=73.0% | Val: R@1=31.5% R@5=41.5% R@10=48.1% R@20=54.3% R@100=74.0%
Embeddings norm: 45.816566467285156, Projected concept

{'100% Correction': {1: 0.30845392231530844,
  5: 0.4143183549124143,
  10: 0.472962680883473,
  20: 0.5407463823305407,
  100: 0.7456207159177456},
 '50% Correction + soft': {1: 0.15536938309215537,
  5: 0.2399086062452399,
  10: 0.28255902513328257,
  20: 0.3396801218583397,
  100: 0.571972581873572},
 '0% Correction': {1: 0.178979436405179,
  5: 0.25894897182025894,
  10: 0.3038842345773039,
  20: 0.36633663366336633,
  100: 0.5575019040365575},
 '0% Correction Hard': {1: 0.25133282559025133,
  5: 0.34424980959634427,
  10: 0.4028941355674029,
  20: 0.46534653465346537,
  100: 0.667936024371668}}